In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load datasets
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Standardizing team names
team_mapping = {
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad"
}

# Preprocess Matches Data
matches.columns = matches.columns.str.strip().str.lower()
matches.drop(columns=["match_type", "player_of_match", "target_runs", "target_overs", "method", "umpire1", "umpire2", "season", "city", "date", "toss_winner", "toss_decision", "result", "result_margin", "team1", "team2"], inplace=True)
matches["winner"] = matches["winner"].replace(team_mapping)

venue_mapping = {
    "Punjab Cricket Association Stadium, Mohali": "Mohali",
    "MA Chidambaram Stadium, Chepauk": "Chennai",
    "Rajiv Gandhi International Stadium, Uppal": "Hyderabad"
}
matches["venue_canonical"] = matches["venue"].replace(venue_mapping)
matches = matches[["match_id", "winner", "venue_canonical"]]

# Preprocess Deliveries Data
deliveries.columns = deliveries.columns.str.strip().str.lower()
deliveries = deliveries[["match_id", "inning", "batting_team", "bowling_team", "over", "ball", "total_runs", "is_wicket"]]
deliveries["batting_team"] = deliveries["batting_team"].replace(team_mapping)
deliveries["bowling_team"] = deliveries["bowling_team"].replace(team_mapping)

deliveries["cum_runs"] = deliveries.groupby(["match_id", "inning"])["total_runs"].cumsum()
deliveries["cum_wickets"] = deliveries.groupby(["match_id", "inning"])["is_wicket"].cumsum()
deliveries["overs_completed"] = deliveries["over"] + (deliveries["ball"] - 1) / 6
deliveries["current_run_rate"] = deliveries["cum_runs"] / deliveries["overs_completed"].replace(0, 1)

# Compute Target for Second Innings
first_innings_scores = deliveries[deliveries["inning"] == 1].groupby("match_id")["cum_runs"].max().reset_index()
first_innings_scores["target"] = first_innings_scores["cum_runs"] + 1
deliveries = deliveries.merge(first_innings_scores[["match_id", "target"]], on="match_id", how="left")

# Compute Required Run Rate
second_innings = deliveries[deliveries["inning"] == 2]
second_innings["remaining_overs"] = 20 - second_innings["overs_completed"]
second_innings["required_run_rate"] = (second_innings["target"] - second_innings["cum_runs"]) / second_innings["remaining_overs"].replace(0, 1)
second_innings["required_run_rate"] = second_innings["required_run_rate"].replace([float("inf"), -float("inf")], 0)

# Create Target Variable
final_results = matches[["match_id", "winner", "venue_canonical"]]
second_innings = second_innings.merge(final_results, on="match_id", how="left")
second_innings["win"] = (second_innings["batting_team"] == second_innings["winner"]).astype(int)

# Select Final Features
final_data = second_innings[["match_id", "inning", "cum_runs", "cum_wickets", "current_run_rate", "required_run_rate", "target", "batting_team", "bowling_team", "venue_canonical", "win"]]

# Encode categorical variables
final_data = pd.get_dummies(final_data, columns=["batting_team", "bowling_team", "venue_canonical"], drop_first=True)

# Train-test split
X = final_data.drop(columns=["win", "match_id"])
y = final_data["win"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Baseline Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Evaluate Model Performance
    print(f"\n{name} Performance:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("--------------------------------------------------")


C:\Users\HP\AppData\Local\Temp\ipykernel_19972\2448712024.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  second_innings["remaining_overs"] = 20 - second_innings["overs_completed"]
C:\Users\HP\AppData\Local\Temp\ipykernel_19972\2448712024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  second_innings["required_run_rate"] = (second_innings["target"] - second_innings["cum_runs"]) / second_innings["remaining_overs"].replace(0, 1)
C:\Users\HP\AppData\Local\Temp\ipykernel_19972\2448712024.py:53: Settin

Training Logistic Regression...


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression Performance:
Accuracy: 0.8094
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.79      0.80     12033
           1       0.81      0.83      0.82     13116

    accuracy                           0.81     25149
   macro avg       0.81      0.81      0.81     25149
weighted avg       0.81      0.81      0.81     25149

Confusion Matrix:
[[ 9518  2515]
 [ 2278 10838]]
--------------------------------------------------
Training Random Forest...

Random Forest Performance:
Accuracy: 0.9986
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12033
           1       1.00      1.00      1.00     13116

    accuracy                           1.00     25149
   macro avg       1.00      1.00      1.00     25149
weighted avg       1.00      1.00      1.00     25149

Confusion Matrix:
[[12011    22]
 [   12 13104]]
-----------------------------

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [18:32:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Performance:
Accuracy: 0.9825
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98     12033
           1       0.98      0.99      0.98     13116

    accuracy                           0.98     25149
   macro avg       0.98      0.98      0.98     25149
weighted avg       0.98      0.98      0.98     25149

Confusion Matrix:
[[11764   269]
 [  170 12946]]
--------------------------------------------------


In [2]:
pip install xgboost


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/150.0 MB 528.1 kB/s eta 0:04:44
   ---------------------------------------- 0.5/150.0 MB 528.1 kB/s eta 0:04:44
   ---------------------------------------- 0.5/150.0 MB 528.1 kB/s eta 0:04:44
   ---------------------------------------- 0.8/150.0 MB 472.0 kB/s eta 0:05:17
   ---------------------------------------- 0.8/150.0 MB 472.0 kB/s eta 0:05:17
   ---------------------------------------- 0.8/150.0 MB 472.0 kB/s eta 0:05:17
   ---------------------------------------- 1.0/150.0 MB 454.2 kB/s eta 0:05:28
   ----


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load datasets
matches = pd.read_csv("matches.csv")
deliveries = pd.read_csv("deliveries.csv")

# Standardizing team names
team_mapping = {
    "Kings XI Punjab": "Punjab Kings",
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad"
}

# Preprocess Matches Data
matches.columns = matches.columns.str.strip().str.lower()
matches.drop(columns=["match_type", "player_of_match", "target_runs", "target_overs", "method", "umpire1", "umpire2", "season", "city", "date", "toss_winner", "toss_decision", "result", "result_margin", "team1", "team2"], inplace=True)
matches["winner"] = matches["winner"].replace(team_mapping)

venue_mapping = {
    "Punjab Cricket Association Stadium, Mohali": "Mohali",
    "MA Chidambaram Stadium, Chepauk": "Chennai",
    "Rajiv Gandhi International Stadium, Uppal": "Hyderabad"
}
matches["venue_canonical"] = matches["venue"].replace(venue_mapping)
matches = matches[["match_id", "winner", "venue_canonical"]]

# Preprocess Deliveries Data
deliveries.columns = deliveries.columns.str.strip().str.lower()
deliveries = deliveries[["match_id", "inning", "batting_team", "bowling_team", "over", "ball", "total_runs", "is_wicket"]]
deliveries["batting_team"] = deliveries["batting_team"].replace(team_mapping)
deliveries["bowling_team"] = deliveries["bowling_team"].replace(team_mapping)

deliveries["cum_runs"] = deliveries.groupby(["match_id", "inning"])["total_runs"].cumsum()
deliveries["cum_wickets"] = deliveries.groupby(["match_id", "inning"])["is_wicket"].cumsum()
deliveries["overs_completed"] = deliveries["over"] + (deliveries["ball"] - 1) / 6
deliveries["current_run_rate"] = deliveries["cum_runs"] / deliveries["overs_completed"].replace(0, 1)

# Compute Target for Second Innings
first_innings_scores = deliveries[deliveries["inning"] == 1].groupby("match_id")["cum_runs"].max().reset_index()
first_innings_scores["target"] = first_innings_scores["cum_runs"] + 1
deliveries = deliveries.merge(first_innings_scores[["match_id", "target"]], on="match_id", how="left")

# Compute Required Run Rate
second_innings = deliveries[deliveries["inning"] == 2]
second_innings["remaining_overs"] = 20 - second_innings["overs_completed"]
second_innings["required_run_rate"] = (second_innings["target"] - second_innings["cum_runs"]) / second_innings["remaining_overs"].replace(0, 1)
second_innings["required_run_rate"] = second_innings["required_run_rate"].replace([float("inf"), -float("inf")], 0)

# Create Target Variable
final_results = matches[["match_id", "winner", "venue_canonical"]]
second_innings = second_innings.merge(final_results, on="match_id", how="left")
second_innings["win"] = (second_innings["batting_team"] == second_innings["winner"]).astype(int)

# Select Final Features
final_data = second_innings[["match_id", "inning", "cum_runs", "cum_wickets", "current_run_rate", "required_run_rate", "target", "batting_team", "bowling_team", "venue_canonical", "win"]]

# Encode categorical variables
final_data = pd.get_dummies(final_data, columns=["batting_team", "bowling_team", "venue_canonical"], drop_first=True)

# Train-test split
X = final_data.drop(columns=["win", "match_id"])
y = final_data["win"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train and Tune Models using Grid Search
param_grid_rf = {'n_estimators': [100, 200, 300], 'max_depth': [10, 20, 30]}
param_grid_lr = {'C': [0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
param_grid_xgb = {'n_estimators': [100, 200], 'max_depth': [3, 6, 9], 'learning_rate': [0.01, 0.1, 0.2]}

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), param_grid_lr),
    "Random Forest": (RandomForestClassifier(random_state=42), param_grid_rf),
    "XGBoost": (XGBClassifier(use_label_encoder=False, eval_metric='logloss'), param_grid_xgb)
}

for name, (model, param_grid) in models.items():
    print(f"Tuning and Training {name}...")
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    
    y_pred = best_model.predict(X_test)
    
    # Evaluate Model Performance
    print(f"\n{name} Performance:")
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("--------------------------------------------------")


C:\Users\HP\AppData\Local\Temp\ipykernel_12564\950406141.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  second_innings["remaining_overs"] = 20 - second_innings["overs_completed"]
C:\Users\HP\AppData\Local\Temp\ipykernel_12564\950406141.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  second_innings["required_run_rate"] = (second_innings["target"] - second_innings["cum_runs"]) / second_innings["remaining_overs"].replace(0, 1)
C:\Users\HP\AppData\Local\Temp\ipykernel_12564\950406141.py:53: SettingWi

Tuning and Training Logistic Regression...

Logistic Regression Performance:
Best Parameters: {'C': 0.1, 'solver': 'liblinear'}
Accuracy: 0.8110
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.79      0.80     12033
           1       0.81      0.83      0.82     13116

    accuracy                           0.81     25149
   macro avg       0.81      0.81      0.81     25149
weighted avg       0.81      0.81      0.81     25149

Confusion Matrix:
[[ 9537  2496]
 [ 2257 10859]]
--------------------------------------------------
Tuning and Training Random Forest...

Random Forest Performance:
Best Parameters: {'max_depth': 30, 'n_estimators': 300}
Accuracy: 0.9969
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12033
           1       1.00      1.00      1.00     13116

    accuracy                           1.00     25149
   macro avg       1.00   

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:183: UserWarning: [18:54:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Performance:
Best Parameters: {'learning_rate': 0.2, 'max_depth': 9, 'n_estimators': 200}
Accuracy: 0.9993
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     12033
           1       1.00      1.00      1.00     13116

    accuracy                           1.00     25149
   macro avg       1.00      1.00      1.00     25149
weighted avg       1.00      1.00      1.00     25149

Confusion Matrix:
[[12021    12]
 [    5 13111]]
--------------------------------------------------
